# TUGAS MANDIRI PRAKTIKUM BIG DATA — PERTEMUAN 12
## Klasifikasi & ML Pipeline dengan Spark MLlib

| Informasi | Detail |
|---|---|
| **Mata Kuliah** | Praktikum Big Data |
| **Pertemuan** | 12 (Tugas Mandiri) |
| **Nama Mahasiswa** | Yarsya |
| **NPM** | 2505060066 |
| **Program Studi** | S1 Teknologi Informasi — Universitas Tidar |

---

### Skenario Tugas
Manajemen **"Kopi Nusantara"** ingin membangun model klasifikasi prediktif untuk memperkirakan apakah pesanan pelanggan berpotensi menghasilkan **ulasan positif (`ulasan_positif = 1`, rating >= 4) atau tidak (`0`)** berdasarkan jenis menu, cabang, waktu pesan, jumlah item (`qty`), dan lama waktu tunggu penyajian (`waktu_tunggu_menit`). Seluruh alur *feature engineering* hingga pemodelan dikemas ke dalam **Spark ML Pipeline** yang membandingkan algoritma **Logistic Regression** dan **Random Forest**, diikuti analisis kesalahan prediksi (*error analysis*) serta perumusan rekomendasi bisnis.

## 0. Inisialisasi SparkSession & Persiapan Dataset
Menyiapkan `SparkSession`, membangkitkan dataset `tugas12_ulasan.csv` (3.000 baris), dan membacanya ke dalam Spark DataFrame.

In [1]:
import os
import sys
import numpy as np
import pandas as pd

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, min as spark_min, max as spark_max, count
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

spark = SparkSession.builder \
    .appName("Tugas12-MLPipeline-Yarsya") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Membangkitkan dataset tugas12_ulasan.csv
np.random.seed(212)
n = 3000
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas12 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "waktu_tunggu_menit": np.random.randint(2, 25, size=n),
})

# Simulasi peluang ulasan positif menurun seiring lamanya waktu tunggu
prob_positif = 1 / (1 + np.exp((data_tugas12["waktu_tunggu_menit"] - 12) / 4))
data_tugas12["ulasan_positif"] = (np.random.rand(n) < prob_positif).astype(int)

data_tugas12.to_csv("tugas12_ulasan.csv", index=False)

# Membaca dataset dengan Spark
df = spark.read.csv("tugas12_ulasan.csv", header=True, inferSchema=True)
print(f"Dataset berhasil dimuat: {df.count()} baris")
df.printSchema()
print("Distribusi label ulasan_positif:")
df.groupBy("ulasan_positif").count().orderBy("ulasan_positif").show()
df.show(5)

ERROR StatusLogger Could not determine local host name
 java.net.UnknownHostException: sandbox: sandbox: Temporary failure in name resolution
	at java.base/java.net.InetAddress.getLocalHost(InetAddress.java:1670)
	at org.apache.logging.log4j.core.util.NetUtils.getLocalHostname(NetUtils.java:56)
	at org.apache.logging.log4j.core.LoggerContext.lambda$setConfiguration$0(LoggerContext.java:615)
	at java.base/java.util.concurrent.ConcurrentHashMap.computeIfAbsent(ConcurrentHashMap.java:1705)
	at org.apache.logging.log4j.core.LoggerContext.setConfiguration(LoggerContext.java:615)
	at org.apache.logging.log4j.core.LoggerContext.reconfigure(LoggerContext.java:694)
	at org.apache.logging.log4j.core.LoggerContext.reconfigure(LoggerContext.java:711)
	at org.apache.logging.log4j.core.LoggerContext.start(LoggerContext.java:253)
	at org.apache.logging.log4j.core.impl.Log4jContextFactory.getContext(Log4jContextFactory.java:245)
	at org.apache.logging.log4j.core.impl.Log4jContextFactory.getContext(Log

ERROR StatusConsoleListener Could not determine local host name
 java.net.UnknownHostException: sandbox: sandbox: Temporary failure in name resolution
	at java.base/java.net.InetAddress.getLocalHost(InetAddress.java:1670)
	at org.apache.logging.log4j.core.util.NetUtils.getLocalHostname(NetUtils.java:56)
	at org.apache.logging.log4j.core.LoggerContext.lambda$setConfiguration$0(LoggerContext.java:615)
	at java.base/java.util.concurrent.ConcurrentHashMap.computeIfAbsent(ConcurrentHashMap.java:1705)
	at org.apache.logging.log4j.core.LoggerContext.setConfiguration(LoggerContext.java:615)
	at org.apache.logging.log4j.core.LoggerContext.reconfigure(LoggerContext.java:694)
	at org.apache.logging.log4j.core.LoggerContext.setConfigLocation(LoggerContext.java:679)
	at org.apache.spark.internal.Logging.initializeLogging(Logging.scala:139)
	at org.apache.spark.internal.Logging.initializeLogIfNecessary(Logging.scala:114)
	at org.apache.spark.internal.Logging.initializeLogIfNecessary$(Logging.scala:1

ERROR StatusConsoleListener Could not determine local host name
 java.net.UnknownHostException: sandbox: sandbox: Temporary failure in name resolution
	at java.base/java.net.InetAddress.getLocalHost(InetAddress.java:1670)
	at org.apache.logging.log4j.core.util.NetUtils.getLocalHostname(NetUtils.java:56)
	at org.apache.logging.log4j.core.LoggerContext.lambda$setConfiguration$0(LoggerContext.java:615)
	at java.base/java.util.concurrent.ConcurrentHashMap.computeIfAbsent(ConcurrentHashMap.java:1705)
	at org.apache.logging.log4j.core.LoggerContext.setConfiguration(LoggerContext.java:615)
	at org.apache.logging.log4j.core.LoggerContext.reconfigure(LoggerContext.java:694)
	at org.apache.logging.log4j.core.LoggerContext.reconfigure(LoggerContext.java:711)
	at org.apache.spark.internal.Logging$.uninitialize(Logging.scala:220)
	at org.apache.spark.deploy.SparkSubmit.org$apache$spark$deploy$SparkSubmit$$runMain(SparkSubmit.scala:972)
	at org.apache.spark.deploy.SparkSubmit.doRunMain$1(SparkSubmit

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/26 16:11:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/11/26 16:11:39 WARN MacAddressUtil: Failed to find a usable hardware address from the network interfaces; using random bytes: 56:d8:84:33:5d:5c:a2:90


Dataset berhasil dimuat: 3000 baris
root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- waktu_tunggu_menit: integer (nullable = true)
 |-- ulasan_positif: integer (nullable = true)

Distribusi label ulasan_positif:


+--------------+-----+
|ulasan_positif|count|
+--------------+-----+
|             0| 1610|
|             1| 1390|
+--------------+-----+

+--------------+----------+-----------+---+------------------+--------------+
|     nama_menu|    cabang|waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|
+--------------+----------+-----------+---+------------------+--------------+
|     Kopi Susu|Yogyakarta|       Sore|  5|                 5|             1|
|        Es Teh|Yogyakarta|       Pagi|  1|                21|             0|
|    Cappuccino|Yogyakarta|       Sore|  5|                15|             0|
|Kentang Goreng|  Semarang|      Malam|  2|                 5|             1|
|     Americano|Yogyakarta|      Malam|  2|                10|             0|
+--------------+----------+-----------+---+------------------+--------------+
only showing top 5 rows



---
## Bagian A. Membangun ML Pipeline (Bobot 25%)

Merangkai tahapan *preprocessing* dan model ke dalam satu `Pipeline`:
1. `StringIndexer` untuk `nama_menu`, `cabang`, dan `waktu_pesan`.
2. `OneHotEncoder` untuk ketiga kolom hasil indexing.
3. `VectorAssembler` menggabungkan hasil one-hot encoding dengan `qty` dan `waktu_tunggu_menit` ke kolom `features`.
4. `LogisticRegression` dengan `labelCol="ulasan_positif"`.

In [2]:
# Tahap 1-3: StringIndexer
idx_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx")
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
idx_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx")

# Tahap 4: OneHotEncoder
ohe = OneHotEncoder(
    inputCols=["nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCols=["nama_menu_vec", "cabang_vec", "waktu_pesan_vec"]
)

# Tahap 5: VectorAssembler
assembler = VectorAssembler(
    inputCols=["qty", "waktu_tunggu_menit", "nama_menu_vec", "cabang_vec", "waktu_pesan_vec"],
    outputCol="features"
)

# Tahap 6: LogisticRegression
lr = LogisticRegression(featuresCol="features", labelCol="ulasan_positif")

# Merangkai menjadi Pipeline
pipeline_lr = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe, assembler, lr])
print("ML Pipeline (Logistic Regression) dengan 6 tahap berhasil didefinisikan!")

ML Pipeline (Logistic Regression) dengan 6 tahap berhasil didefinisikan!


---
## Bagian B. Pelatihan & Evaluasi Logistic Regression (Bobot 25%)

Membagi data **80:20** (`seed=77`), melatih pipeline Logistic Regression, lalu mengevaluasi pada data testing menggunakan **Accuracy**, **F1-score**, **AUC-ROC**, serta menampilkan **Confusion Matrix** sederhana via `groupBy`.

In [3]:
# Split data 80:20 dengan seed=77
train_data, test_data = df.randomSplit([0.8, 0.2], seed=77)
print(f"Data training: {train_data.count()} baris | Data testing: {test_data.count()} baris")

# Melatih Pipeline Logistic Regression
model_lr = pipeline_lr.fit(train_data)
pred_lr = model_lr.transform(test_data)

# Evaluator untuk Accuracy, F1-score, dan AUC-ROC
ev_acc = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="accuracy")
ev_f1 = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="f1")
ev_auc = BinaryClassificationEvaluator(labelCol="ulasan_positif", metricName="areaUnderROC")

acc_lr = ev_acc.evaluate(pred_lr)
f1_lr = ev_f1.evaluate(pred_lr)
auc_lr = ev_auc.evaluate(pred_lr)

print("\n=== EVALUASI LOGISTIC REGRESSION ===")
print(f"Accuracy : {acc_lr:.4f}")
print(f"F1-score : {f1_lr:.4f}")
print(f"AUC-ROC  : {auc_lr:.4f}")

print("\n=== CONFUSION MATRIX (LOGISTIC REGRESSION) ===")
pred_lr.groupBy("ulasan_positif", "prediction").count().orderBy("ulasan_positif", "prediction").show()

Data training: 2411 baris | Data testing: 589 baris



=== EVALUASI LOGISTIC REGRESSION ===
Accuracy : 0.7929
F1-score : 0.7926
AUC-ROC  : 0.8650

=== CONFUSION MATRIX (LOGISTIC REGRESSION) ===


+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  257|
|             0|       1.0|   57|
|             1|       0.0|   65|
|             1|       1.0|  210|
+--------------+----------+-----+



---
## Bagian C. Model Pembanding — RandomForestClassifier (Bobot 20%)

Melatih `RandomForestClassifier` (`numTrees=30`) menggunakan 5 tahap *feature engineering* yang identik dari Bagian A, kemudian membandingkan metrik Accuracy, F1-score, dan AUC-ROC dengan model Logistic Regression.

In [4]:
# Model pembanding: RandomForestClassifier dengan numTrees=30
rf = RandomForestClassifier(featuresCol="features", labelCol="ulasan_positif", numTrees=30, seed=77)

# Pipeline baru dengan 5 tahap feature engineering yang identik
pipeline_rf = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe, assembler, rf])
model_rf = pipeline_rf.fit(train_data)
pred_rf = model_rf.transform(test_data)

acc_rf = ev_acc.evaluate(pred_rf)
f1_rf = ev_f1.evaluate(pred_rf)
auc_rf = ev_auc.evaluate(pred_rf)

print("=== EVALUASI RANDOM FOREST (numTrees=30) ===")
print(f"Accuracy : {acc_rf:.4f}")
print(f"F1-score : {f1_rf:.4f}")
print(f"AUC-ROC  : {auc_rf:.4f}")

print("\n=== CONFUSION MATRIX (RANDOM FOREST) ===")
pred_rf.groupBy("ulasan_positif", "prediction").count().orderBy("ulasan_positif", "prediction").show()

print("=== PERBANDINGAN METRIK KEDUA MODEL ===")
print(f"Logistic Regression -> Accuracy: {acc_lr:.4f} | F1-score: {f1_lr:.4f} | AUC-ROC: {auc_lr:.4f}")
print(f"Random Forest       -> Accuracy: {acc_rf:.4f} | F1-score: {f1_rf:.4f} | AUC-ROC: {auc_rf:.4f}")

=== EVALUASI RANDOM FOREST (numTrees=30) ===
Accuracy : 0.7946
F1-score : 0.7943
AUC-ROC  : 0.8555

=== CONFUSION MATRIX (RANDOM FOREST) ===
+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  258|
|             0|       1.0|   56|
|             1|       0.0|   65|
|             1|       1.0|  210|
+--------------+----------+-----+

=== PERBANDINGAN METRIK KEDUA MODEL ===
Logistic Regression -> Accuracy: 0.7929 | F1-score: 0.7926 | AUC-ROC: 0.8650
Random Forest       -> Accuracy: 0.7946 | F1-score: 0.7943 | AUC-ROC: 0.8555


---
## Bagian D. Analisis Kesalahan Model (Bobot 15%)

Memilih model terbaik berdasarkan nilai **AUC-ROC tertinggi**, lalu menampilkan 10 baris di mana model **salah memprediksi** (`ulasan_positif != prediction`) beserta ringkasan pola `waktu_tunggu_menit` pada kasus-kasus kesalahan tersebut.

In [5]:
# Memilih model terbaik berdasarkan AUC tertinggi
if auc_lr >= auc_rf:
    nama_terbaik = "Logistic Regression"
    pred_terbaik = pred_lr
else:
    nama_terbaik = "Random Forest"
    pred_terbaik = pred_rf

print(f"Model terbaik berdasarkan AUC tertinggi: {nama_terbaik}")

# Filter baris yang salah prediksi (label != prediction)
kesalahan_df = pred_terbaik.filter(col("ulasan_positif") != col("prediction"))
print(f"Total kesalahan prediksi pada data test: {kesalahan_df.count()} dari {test_data.count()} baris")

print("\n=== 10 BARIS SAMPEL KESALAHAN PREDIKSI ===")
kesalahan_df.select("nama_menu", "cabang", "waktu_pesan", "qty", "waktu_tunggu_menit", "ulasan_positif", "prediction", "probability").show(10, truncate=False)

print("=== STATISTIK WAKTU TUNGGU PADA BARIS SALAH PREDIKSI ===")
kesalahan_df.agg(
    spark_min("waktu_tunggu_menit").alias("min_waktu_tunggu"),
    avg("waktu_tunggu_menit").alias("rata2_waktu_tunggu"),
    spark_max("waktu_tunggu_menit").alias("max_waktu_tunggu")
).show()

print("Distribusi waktu_tunggu_menit teratas pada kesalahan prediksi:")
kesalahan_df.groupBy("waktu_tunggu_menit").count().orderBy(col("count").desc()).show(8)

Model terbaik berdasarkan AUC tertinggi: Logistic Regression


Total kesalahan prediksi pada data test: 122 dari 589 baris

=== 10 BARIS SAMPEL KESALAHAN PREDIKSI ===
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|nama_menu|cabang  |waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|probability                             |
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|Americano|Magelang|Malam      |1  |15                |1             |0.0       |[0.6253879175414356,0.37461208245856437]|
|Americano|Magelang|Siang      |2  |22                |1             |0.0       |[0.9113822566149019,0.08861774338509809]|
|Americano|Magelang|Sore       |1  |7                 |0             |1.0       |[0.23209137939008503,0.7679086206099149]|
|Americano|Magelang|Sore       |2  |11                |0             |1.0       |[0.42615454016906074,0.5738454598309393]|
|Americano|Magelang|Sore       |3  

+----------------+------------------+----------------+
|min_waktu_tunggu|rata2_waktu_tunggu|max_waktu_tunggu|
+----------------+------------------+----------------+
|               2|12.647540983606557|              24|
+----------------+------------------+----------------+

Distribusi waktu_tunggu_menit teratas pada kesalahan prediksi:


+------------------+-----+
|waktu_tunggu_menit|count|
+------------------+-----+
|                14|   13|
|                11|   11|
|                12|   10|
|                13|   10|
|                16|    8|
|                10|    8|
|                 8|    7|
|                17|    6|
+------------------+-----+
only showing top 8 rows



### Pengamatan Pola Kesalahan (Bagian D)
Berdasarkan keluaran di atas, terlihat pola kesalahan prediksi yang sangat jelas:
- **Konsentrasi pada Zona Ambang Batas (*Decision Boundary*):** Mayoritas kesalahan prediksi terjadi ketika `waktu_tunggu_menit` berada di sekitar rentang tengah **9 hingga 15 menit** (sekitar rata-rata 12 menit).
- **Alasan Probabilistik:** Pada rentang waktu tunggu ~12 menit, probabilitas pelanggan memberikan ulasan positif secara alami berada di kisaran **50% (`0.50`)**, sehingga perilaku pelanggan sangat bervariasi antara puas dan tidak puas akibat faktor eksternal yang tidak terekam di data (seperti suasana hati pelanggan atau keramahan staf). Akibatnya, model kesulitan memisahkan kelas 0 dan 1 secara tegas pada area batas tersebut dibandingkan pada waktu tunggu ekstrem (< 5 menit atau > 20 menit).

---
## Bagian E. Kesimpulan Bisnis (Bobot 15%)

Berdasarkan hasil pemodelan klasifikasi dengan Spark ML Pipeline serta analisis kesalahan pada bagian D, rekomendasi konkret berikut dapat diberikan kepada manajemen **"Kopi Nusantara"** untuk meningkatkan kepuasan pelanggan:

1. **Menjadikan `waktu_tunggu_menit` sebagai KPI Operasional Utama:**
   Fitur yang paling dominan memengaruhi prediksi ulasan positif adalah `waktu_tunggu_menit` — semakin lama waktu penyajian, semakin drastis turunnya peluang pelanggan memberi rating >= 4. Manajemen disarankan menetapkan **Standar Layanan Maksimal (SLA) waktu penyajian di bawah 10 menit** untuk menjaga posisi pesanan tetap aman di zona probabilitas ulasan positif yang tinggi.

2. **Intervensi Proaktif pada Zona Rawan (10–15 Menit):**
   Analisis kesalahan menunjukkan bahwa rentang waktu tunggu 10–15 menit merupakan titik krusial yang menentukan kepuasan pelanggan. Ketika antrean di kasir atau dapur mulai menyebabkan estimasi tunggu melewati 10 menit (khususnya saat jam sibuk seperti *Siang* atau *Sore*), barista perlu menerapkan strategi mitigasi proaktif, misalnya memberikan pemberitahuan estimasi waktu secara transparan, menawarkan air mineral gratis, atau memberikan *voucher* potongan tipis untuk kunjungan berikutnya agar pelanggan tetap merasa dihargai.

3. **Optimalisasi Alur Persiapan Menu & Penjadwalan Barista:**
   Karena model menunjukkan ketidakpuasan lebih sering muncul akibat keterlambatan penyajian dibandingkan jenis menu atau lokasi cabang tertentu, manajemen dapat memanfaatkan hasil klasifikasi *real-time* sebagai sistem peringatan dini (*early warning*) pada layar dapur (*Kitchen Display System*) untuk memprioritaskan pesanan dengan `qty` besar agar tidak melewati ambang kritis 12 menit.

In [6]:
# Menutup SparkSession
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
